# Intelligent Hospital Workflow Assistant
## Deep Learning Project — Milestone 2



---
## Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)

Project root: /Users/ramezhany/Uni/Sem 10/Deep Learning/Project/DLCV-Milestone 2


## Imports

In [ ]:
from dotenv import load_dotenv
import os
from backend.database import engine
from backend.models import Base
from backend.seed_data import seed
from backend.database import get_session
from backend.models import Doctor
from backend.models import User
from backend.tools import (
    get_patient_history,
    find_available_doctor,
    book_appointment,
    save_medical_report,
    send_confirmation_email,
)
import json
from backend.agent import build_agent_executor
from backend.tools import get_doctor_slots, get_all_appointments

load_dotenv()

assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set — edit .env first'
print('API key loaded.')

API key loaded.


In [ ]:


Base.metadata.create_all(bind=engine)
seed()

Database already seeded. Skipping.


In [ ]:


db = get_session()
doctors = db.query(Doctor).all()
db.close()

print(f'{len(doctors)} doctors seeded:')
for d in doctors:
    print(f'  [{d.id}] {d.name} — {d.specialty} ({d.department})')

8 doctors seeded:
  [1] Dr. Sarah Mitchell — cardiology (Cardiology)
  [2] Dr. James Okafor — neurology (Neurology)
  [3] Dr. Lena Hoffmann — general (General Practice)
  [4] Dr. Ahmed Al-Rashid — orthopedics (Orthopedics)
  [5] Dr. Priya Sharma — dermatology (Dermatology)
  [6] Dr. Carlos Mendes — gastroenterology (Gastroenterology)
  [7] Dr. Fatima Zahra — pulmonology (Pulmonology)
  [8] Dr. Wei Chen — endocrinology (Endocrinology)


In [ ]:


db = get_session()
for email, age, history in [
    ('ramezhany986@gmail.com', 45, 'Type 2 diabetes, hypertension'),
    ('mohamedshady2003@gmail.com', 28, 'None'),
]:
    if not db.query(User).filter(User.email == email).first():
        db.add(User(email=email, age=age, chronic_disease_history=history))
db.commit()
db.close()
print('Test patients registered.')

Test patients registered.



##  Tool Demos

In [ ]:


def pretty(result):
    print(json.dumps(result, indent=2, default=str))

In [ ]:
print('=== Tool 1: get_patient_history ===')
pretty(get_patient_history.invoke({'email': 'ramezhany986@gmail.com'}))

=== Tool 1: get_patient_history ===
{
  "success": true,
  "data": {
    "email": "ramezhany986@gmail.com",
    "age": 45,
    "chronic_disease_history": "Type 2 diabetes, hypertension",
    "past_appointments": [
      {
        "appointment_id": 3,
        "doctor": "Dr. Sarah Mitchell",
        "specialty": "cardiology",
        "slot": "2026-04-25 09:00",
        "priority": 0,
        "report_summary": "URGENT - Priority 0 Triage Assessment. Patient: Male/Female, 45 years old. Known history of Type 2 diabetes and hypertension \u2014 both significant cardiovascular risk factors. Chief Complaint: Severe chest pain radiating to the left arm, accompanied by difficulty breathing. Onset: Approximately 20 minutes prior to intake. Symptom profile is consistent with a potential acute myocardial infarction (AMI) or other serious acute coronary syndrome. Immediate cardiology evaluation is required. Patient was triaged as Priority 0 (URGENT) and routed to Cardiology without delay. NOTE: This 

In [ ]:
print('=== Tool 2: find_available_doctor (cardiology) ===')
doctor_result = find_available_doctor.invoke({'specialty': 'Dermatology'})
pretty(doctor_result)

=== Tool 2: find_available_doctor (cardiology) ===
{
  "success": true,
  "data": {
    "doctor_id": 5,
    "doctor_name": "Dr. Priya Sharma",
    "specialty": "dermatology",
    "department": "Dermatology",
    "slot_id": 41,
    "slot_datetime": "2026-04-25 09:00"
  }
}


In [ ]:
print('=== Tool 3: book_appointment ===')
d = doctor_result['data']
booking = book_appointment.invoke({
    'user_email': 'alice@example.com',
    'doctor_id': d['doctor_id'],
    'slot_id': d['slot_id'],
    'priority': 1,
})
pretty(booking)

=== Tool 3: book_appointment ===
{
  "success": true,
  "data": {
    "appointment_id": 1,
    "doctor_name": "Dr. Priya Sharma",
    "specialty": "dermatology",
    "slot_datetime": "2026-04-25 09:00",
    "priority": 1
  }
}


In [ ]:
print('=== Tool 4: save_medical_report ===')
appt_id = booking['data']['appointment_id']
pretty(save_medical_report.invoke({
    'appointment_id': appt_id,
    'summary': 'Patient reports persistent chest tightness for 3 days. Known hypertension. Triage: Priority 1.',
    'medication_recommendations': 'Continue antihypertensive medication. Avoid strenuous activity.',
}))

=== Tool 4: save_medical_report ===
{
  "success": true,
  "data": {
    "report_id": 1,
    "appointment_id": 1
  }
}


In [ ]:
print('=== Tool 5: send_confirmation_email (logged, no SMTP configured) ===')
pretty(send_confirmation_email.invoke({
    'to_email': 'ramezhany986@gmail.com',
    'doctor_name': d['doctor_name'],
    'slot_datetime': d['slot_datetime'],
    'department': d['department'],
}))

=== Tool 5: send_confirmation_email (logged, no SMTP configured) ===
{
  "success": true,
  "data": {
    "sent_to": "ramezhany986@gmail.com",
    "method": "smtp"
  }
}



## Agent Demo

In [ ]:


executor = build_agent_executor()

/Users/ramezhany/Uni/Sem 10/Deep Learning/Project/DLCV-Milestone 2/backend/agent.py:97: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = session_memory or ConversationBufferMemory(


In [ ]:
print('='*60)
print('SCENARIO 1: Urgent — chest pain + shortness of breath (Priority 0)')
print('='*60)

r = executor.invoke({
    'input': 'Patient email: ramezhany986@gmail.com\n\nPatient message: I have severe chest pain radiating to my left arm and I can barely breathe. It started 20 minutes ago.'
})
print('\n--- AGENT FINAL RESPONSE ---')
print(r['output'])

SCENARIO 1: Urgent — chest pain + shortness of breath (Priority 0)


> Entering new AgentExecutor chain...
Thought: This is a Priority 0 URGENT case. The patient is describing classic heart attack symptoms - severe chest pain radiating to the left arm combined with difficulty breathing, starting 20 minutes ago. I must act immediately without asking follow-up questions. Let me start by retrieving the patient's history.

Action: get_patient_history
Action Input: ramezhany986@gmail.com{'success': True, 'data': {'email': 'ramezhany986@gmail.com', 'age': 45, 'chronic_disease_history': 'Type 2 diabetes, hypertension', 'past_appointments': [{'appointment_id': 3, 'doctor': 'Dr. Sarah Mitchell', 'specialty': 'cardiology', 'slot': '2026-04-25 09:00', 'priority': 0, 'report_summary': 'URGENT - Priority 0 Triage Assessment. Patient: Male/Female, 45 years old. Known history of Type 2 diabetes and hypertension — both significant cardiovascular risk factors. Chief Complaint: Severe chest pain radiati

In [ ]:
print('='*60)
print('SCENARIO 2: Multi-turn — vague symptoms requiring follow-up')
print('='*60)

executor2 = build_agent_executor()

r1 = executor2.invoke({'input': 'Patient email: alice@example.com\n\nPatient message: I have been feeling dizzy and tired lately.'})
print('[AGENT TURN 1]:', r1['output'])

r2 = executor2.invoke({'input': 'Patient email: alice@example.com\n\nPatient message: About 5 days. The dizziness is worst in the morning and I also have a headache.'})
print('\n[AGENT TURN 2]:', r2['output'])

In [ ]:
print('='*60)
print('SCENARIO 3: Routine — prescription renewal (Priority 3)')
print('='*60)

executor3 = build_agent_executor()
r = executor3.invoke({'input': 'Patient email: bob@example.com\n\nPatient message: I need to renew my asthma inhaler prescription. No new symptoms, just running out.'})
print('[AGENT RESPONSE]:', r['output'])

In [ ]:
get_doctor_slots.invoke({"doctor_id": "1"})

{'success': True,
 'data': {'doctor_id': 1,
  'doctor_name': 'Dr. Sarah Mitchell',
  'specialty': 'cardiology',
  'slots': [{'slot_id': 1, 'datetime': '2026-04-25 09:00', 'status': 'booked'},
   {'slot_id': 2, 'datetime': '2026-04-25 11:00', 'status': 'available'},
   {'slot_id': 3, 'datetime': '2026-04-26 09:00', 'status': 'available'},
   {'slot_id': 4, 'datetime': '2026-04-26 11:00', 'status': 'available'},
   {'slot_id': 5, 'datetime': '2026-04-27 09:00', 'status': 'available'},
   {'slot_id': 6, 'datetime': '2026-04-27 11:00', 'status': 'available'},
   {'slot_id': 7, 'datetime': '2026-04-28 09:00', 'status': 'available'},
   {'slot_id': 8, 'datetime': '2026-04-28 11:00', 'status': 'available'},
   {'slot_id': 9, 'datetime': '2026-04-29 09:00', 'status': 'available'},
   {'slot_id': 10, 'datetime': '2026-04-29 11:00', 'status': 'available'}]}}

In [ ]:
get_all_appointments.invoke({"_": ""})

{'success': True,
 'data': [{'appointment_id': 1,
   'patient_email': 'alice@example.com',
   'doctor': 'Dr. Priya Sharma',
   'specialty': 'dermatology',
   'slot_datetime': '2026-04-25 09:00',
   'priority': 1,
   'has_report': True},
  {'appointment_id': 2,
   'patient_email': 'bob@example.com',
   'doctor': 'Dr. James Okafor',
   'specialty': 'neurology',
   'slot_datetime': '2026-04-25 09:00',
   'priority': 0,
   'has_report': False},
  {'appointment_id': 3,
   'patient_email': 'ramezhany986@gmail.com',
   'doctor': 'Dr. Sarah Mitchell',
   'specialty': 'cardiology',
   'slot_datetime': '2026-04-25 09:00',
   'priority': 0,
   'has_report': True}]}

In [ ]:
import re

TEST_CASES = [
    ('Severe chest pain radiating to left arm, started 10 minutes ago', 0),
    ('Sudden loss of speech and facial drooping on the right side', 0),
    ('High fever 39.5 degrees for two days with severe headache', 1),
    ('Moderate knee pain after a fall, can still walk', 2),
    ('Dry cough and mild fatigue for the past week', 2),
    ('Routine blood pressure check, no symptoms', 3),
    ('Prescription renewal for daily metformin', 3),
]

results = []
for desc, expected in TEST_CASES:
    ex = build_agent_executor()
    resp = ex.invoke({'input': f'Patient email: bob@example.com\n\nPatient message: {desc}'})
    output = resp['output']
    match = re.search(r'[Pp]riority\s*[:\-]?\s*(\d)', output)
    predicted = int(match.group(1)) if match else -1
    correct = (predicted == expected)
    results.append((desc[:55], expected, predicted, correct))
    print(f'[{"PASS" if correct else "FAIL"}] Expected P{expected}, Got P{predicted} | {desc[:55]}')

accuracy = sum(r[3] for r in results) / len(results)
print(f'\nAccuracy: {accuracy:.0%} ({sum(r[3] for r in results)}/{len(results)})')

### Priority Triage Accuracy — Results Table


In [46]:
import pandas as pd

PRIORITY_LABELS = {0: 'P0 — Urgent', 1: 'P1 — High', 2: 'P2 — Moderate', 3: 'P3 — Routine', -1: 'N/A (unregistered)'}

rows = [
    ('Severe chest pain radiating to left arm, started 10 min ago', 0, -1, False,  'Booking FK error — unregistered patient; priority not extractable'),
    ('Sudden loss of speech and facial drooping on the right side',  0,  0, True,   'Correctly identified stroke → P0'),
    ('High fever 39.5°C for two days with severe headache',          1,  1, True,   'Correctly identified high-fever + headache → P1'),
    ('Moderate knee pain after a fall, can still walk',              2,  2, True,   'Correctly identified orthopedic trauma → P2'),
    ('Dry cough and mild fatigue for the past week',                 2,  3, False,  'Misclassified: mild + 1-week duration → P3 (expected P2)'),
    ('Routine blood pressure check, no symptoms',                    3,  3, True,   'Correctly identified routine checkup → P3'),
    ('Prescription renewal for daily metformin',                     3,  3, True,   'Correctly identified prescription renewal → P3'),
]

df = pd.DataFrame(rows, columns=['Symptom Description', 'Expected', 'Predicted', 'Correct', 'Notes'])
df['Expected']  = df['Expected'].map(PRIORITY_LABELS)
df['Predicted'] = df['Predicted'].map(PRIORITY_LABELS)
df['Correct']   = df['Correct'].map({True: '✅ PASS', False: '❌ FAIL'})
df.index = range(1, len(df) + 1)
df.index.name = '#'

print(f'Priority Triage Accuracy: 5 / 7  =  71%')
print()
df


Priority Triage Accuracy: 5 / 7  =  71%



,Symptom Description,Expected,Predicted,Correct,Notes
#,,,,,
1,"Severe chest pain radiating to left arm, start...",P0 — Urgent,N/A (unregistered),❌ FAIL,Booking FK error — unregistered patient; prior...
2,Sudden loss of speech and facial drooping on t...,P0 — Urgent,P0 — Urgent,✅ PASS,Correctly identified stroke → P0
3,High fever 39.5°C for two days with severe hea...,P1 — High,P1 — High,✅ PASS,Correctly identified high-fever + headache → P1
4,"Moderate knee pain after a fall, can still walk",P2 — Moderate,P2 — Moderate,✅ PASS,Correctly identified orthopedic trauma → P2
5,Dry cough and mild fatigue for the past week,P2 — Moderate,P3 — Routine,❌ FAIL,Misclassified: mild + 1-week duration → P3 (ex...
6,"Routine blood pressure check, no symptoms",P3 — Routine,P3 — Routine,✅ PASS,Correctly identified routine checkup → P3
7,Prescription renewal for daily metformin,P3 — Routine,P3 — Routine,✅ PASS,Correctly identified prescription renewal → P3


### Failure Analysis

| # | Case | Root Cause | Type |
|---|------|------------|------|
| 1 | Severe chest pain (P0) | `bob@example.com` not registered → FK violation → booking fails → agent's Final Answer never states a numeric priority → regex returns −1 | System constraint (not a model error) |
| 5 | Dry cough + mild fatigue (P2) | Agent interpreted "mild" + "1 week" as Routine (P3) rather than Moderate (P2); absent red-flag symptoms biased the model toward the lower tier | Model classification edge case |

**Takeaway:** Case 1 is a data-integrity failure (unregistered user), not a triage reasoning failure — the agent correctly identified P0 and attempted to book urgently. Case 5 is a genuine boundary ambiguity: without additional context (worsening trend, comorbidities) the P2 vs P3 distinction is subjective.


In [47]:
from backend.tools import show_appointments

print('=== Current Appointment State in DB ===')
show_appointments()


=== Current Appointment State in DB ===


,Patient,Doctor,Specialty,Date & Time,Priority,Report
ID,,,,,,
29,ramezhany986@gmail.com,Dr. Sarah Mitchell,Cardiology,2026-04-25 09:00,P0 — Urgent,Yes
30,mohamedshady2003@gmail.com,Dr. James Okafor,Neurology,2026-04-25 09:00,P2 — Moderate,Yes
32,new@example.com,Dr. Priya Sharma,Dermatology,2026-04-25 09:00,P3 — Routine,Yes
